# LLM-like architecture with pyTorch

Most code from chapter 4 from [thelmbook.com](https://www.thelmbook.com/). Sources:
- ...

In [16]:
from tqdm import tqdm   # For progress bars

import torch
import torch.nn as nn
from transformers import AutoTokenizer

import utils
import thelmbook as tlmb

In [2]:
import yaml

with open("params.yaml") as f:
    params = yaml.safe_load(f)['llm']

# define explicitly to reduce chenges to original code
data_url = params['data_url']
emb_dim = params['emb_dim']
num_heads = params['num_heads']
num_blocks = params['num_blocks']
batch_size = params['batch_size']
learning_rate = params['learning_rate']
num_epochs = params['num_epochs']
context_size = params['context_size']

utils.print_params(params)

random_state: 42
data_url: https://www.thelmbook.com/data/news
emb_dim: 128
num_heads: 8
num_blocks: 2
batch_size: 128
learning_rate: 0.001
num_epochs: 1
context_size: 30


In [3]:
tlmb.set_seed(params['random_state'])

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [5]:
# Initialize the tokenizer using Microsoft's Phi-3.5-mini model
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
# Get padding token index for padding shorter sequences
pad_idx = tokenizer.pad_token_id
pad_idx

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


32000

In [25]:
# Set evaluation interval (number of examples after which to perform validation)
# 200,000 examples provides a good balance between training time and monitoring frequency
eval_interval = 200_000
examples_processed = 0  # Counter for tracking progress toward next evaluation

# Define test contexts for generating sample text during evaluation
contexts = [
    "Moscow",
    "New York",
    "A hurricane",
    "The President"
]

## Dataset

### Load

In [7]:
train_dataloader, test_dataloader = tlmb.download_and_prepare_data(
    params['data_url'], batch_size, tokenizer, context_size
)


./data/news.tar.gz already downloaded.
./data\news ./data\news\train.txt ./data\news\test.txt

Data files already extracted.

Counting sentences in ./data\news\train.txt...

Found 22034911 sentences in ./data\news\train.txt.

Counting sentences in ./data\news\test.txt...

Found 449693 sentences in ./data\news\test.txt.

Training sentences: 22034911

Test sentences: 449693


In [8]:
vocab_size = len(tokenizer)
print(f"\nVocabulary size: {vocab_size}\n")


Vocabulary size: 32011



### Prepare

In [9]:
# code here

## Model

### Build (architecture)

In [ ]:
model = tlmb.DecoderLanguageModel(vocab_size, emb_dim, num_heads, num_blocks, pad_idx)

DecoderLanguageModel(
  (embedding): Embedding(32011, 128, padding_idx=32000)
  (layers): ModuleList(
    (0-1): 2 x DecoderBlock(
      (norm1): RMSNorm()
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x AttentionHead()
        )
      )
      (norm2): RMSNorm()
      (mlp): MLP()
    )
  )
)


In [11]:
model.to(device)

DecoderLanguageModel(
  (embedding): Embedding(32011, 128, padding_idx=32000)
  (layers): ModuleList(
    (0-1): 2 x DecoderBlock(
      (norm1): RMSNorm()
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x AttentionHead()
        )
      )
      (norm2): RMSNorm()
      (mlp): MLP()
    )
  )
)

### Compile

In [13]:
# Initialize model weights using custom initialization scheme
# This is important for stable training of deep neural networks
tlmb.initialize_weights(model)

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [17]:
# Initialize the loss function (Cross Entropy) for training
# ignore_index=pad_idx ensures that padding tokens don't contribute to the loss
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

In [24]:
# Calculate and display the total number of trainable parameters in the model
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params / 1e6}E6\n")


Total trainable parameters: 8.589824E6



### Training loop (fit)

In [ ]:
# code here

## Results

### Predictions

In [ ]:
# code here

### Evaluation (accuracy)

In [ ]:
# code here

### Show

In [ ]:
# code here

Plot wrongs